
# Module 4: Data Preprocessing

## Objective

The objective of this module is to prepare the feature-engineered RouteRanker dataset for machine learning.

This module includes handling missing values, encoding categorical features, applying feature scaling, and preparing the final dataset.

The original dataset will be preserved, and all preprocessing decisions will be documented.



## 1. Importing Libraries and Loading the Dataset

In this step, we import the required Python libraries and load the feature-engineered dataset using Pandas.

The dataset is loaded from the processed data folder of the RouteRanker project.

We will inspect the dataset before applying any preprocessing techniques.


In [1]:

import pandas as pd
import numpy as np
from pathlib import Path

project_dir = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker"
)

data_path = (
    project_dir / "data" / "processed" / "route_features.csv"
)

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

df.head()


Dataset loaded successfully!
Dataset shape: (8322, 20)


,route_id,route_short_name,route_long_name,route_type,scheduled_trip_count,stop_count,estimated_straight_line_route_km,trips_with_distance,operates_monday,operates_tuesday,operates_wednesday,operates_thursday,operates_friday,operates_saturday,operates_sunday,operating_day_count,city,start_service_hour,end_service_hour,trips_with_valid_departure
0,101sh,101sh,P.D.P.U. Cross Road ⇆ Asarva Railway Station,3,10,44,44.528570,10.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,7.0,ahmedabad,6.0,19.0,10
1,103-1,103,Bhakti Circle ⇆ Gujarat University,3,22,80,22.365598,22.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,7.0,ahmedabad,6.0,21.0,22
2,103-2,103,Bhakti Circle ⇆ Sarangpur,3,14,52,14.922204,14.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,7.0,ahmedabad,5.0,23.0,14
3,103-3,103,Gujarat University ⇆ Sarangpur,3,6,28,8.947943,6.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,7.0,ahmedabad,12.0,22.0,6
4,105,105,Naroda Industrial Township( MuthiyaGam ) ⇆ Lal...,3,216,40,63.726733,216.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,7.0,ahmedabad,5.0,23.0,216



## 3. Missing Values and Duplicate Records

Missing values can affect the reliability of machine learning models. In this step, we identify missing values in each column and check for duplicate records.

We will investigate any missing values before deciding whether to recover, impute, or remove them. Duplicate records will also be checked to avoid unnecessary repetition in the dataset.


In [2]:

# Count missing values in each column
missing_values = df.isnull().sum()

print("Missing Values by Column:")
print(missing_values[missing_values > 0])

# Count total missing values
print("\nTotal Missing Values:", df.isnull().sum().sum())

# Check duplicate records
print("\nTotal Duplicate Rows:", df.duplicated().sum())


Missing Values by Column:
route_short_name                    1031
estimated_straight_line_route_km    1010
trips_with_distance                 1010
operates_monday                     1006
operates_tuesday                    1006
operates_wednesday                  1006
operates_thursday                   1006
operates_friday                     1006
operates_saturday                   1006
operates_sunday                     1006
operating_day_count                 1006
start_service_hour                  1006
end_service_hour                    1006
dtype: int64

Total Missing Values: 13111

Total Duplicate Rows: 0



## 4. Investigating Missing Values

In this step, we investigate the distribution of missing values across different cities.

We identify which columns contain missing values and examine whether they are concentrated in particular cities or records.

This investigation helps us choose suitable preprocessing methods while preserving valid information.


In [3]:

# Missing values by city
print("Missing Values by City:")
print(df.groupby("city").apply(
    lambda city_df: city_df.isnull().sum().sum(),
    include_groups=False
))

# Missing values by column and city
missing_by_city = df.groupby("city").apply(
    lambda city_df: city_df.isnull().sum(),
    include_groups=False
)

print("\nMissing Values by Column and City:")
display(missing_by_city)

# Show sample records containing missing values
print("\nSample Records with Missing Values:")
display(df[df.isnull().any(axis=1)].head(10))


Missing Values by City:
city
ahmedabad      72
chennai      8132
hyderabad    1031
mumbai       3876
pune            0
dtype: int64

Missing Values by Column and City:


,route_id,route_short_name,route_long_name,route_type,scheduled_trip_count,stop_count,estimated_straight_line_route_km,trips_with_distance,operates_monday,operates_tuesday,operates_wednesday,operates_thursday,operates_friday,operates_saturday,operates_sunday,operating_day_count,start_service_hour,end_service_hour,trips_with_valid_departure
city,,,,,,,,,,,,,,,,,,,
ahmedabad,0,0,0,0,0,0,6,6,6,6,6,6,6,6,6,6,6,6,0
chennai,0,0,0,0,0,0,681,681,677,677,677,677,677,677,677,677,677,677,0
hyderabad,0,1031,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
mumbai,0,0,0,0,0,0,323,323,323,323,323,323,323,323,323,323,323,323,0
pune,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0



Sample Records with Missing Values:


,route_id,route_short_name,route_long_name,route_type,scheduled_trip_count,stop_count,estimated_straight_line_route_km,trips_with_distance,operates_monday,operates_tuesday,operates_wednesday,operates_thursday,operates_friday,operates_saturday,operates_sunday,operating_day_count,city,start_service_hour,end_service_hour,trips_with_valid_departure
626,Kalol To Pathikashram Via Sardhav,Kalol To Pathikashram Via Sardhav,Kalol To Pathikashram Via Sardhav ⇆ Kalol To P...,3,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ahmedabad,NaN,NaN,0
627,Mansa To Pathikashram,Mansa To Pathikashram,Mansa To Pathikashram ⇆ Mansa To Pathikashram,3,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ahmedabad,NaN,NaN,0
631,Pathikashram To Kalol Via Sardhav,Pathikashram To Kalol Via Sardhav,Pathikashram To Kalol Via Sardhav ⇆ Pathikashr...,3,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ahmedabad,NaN,NaN,0
632,Pathikashram To Mansa,Pathikashram To Mansa,Pathikashram To Mansa ⇆ Pathikashram To Mansa,3,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ahmedabad,NaN,NaN,0
634,Pathikashram To Pundhara,Pathikashram To Pundhara,Pathikashram To Pundhara ⇆ Pathikashram To Pun...,3,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ahmedabad,NaN,NaN,0
637,Pundhara To Pathikashram,Pundhara To Pathikashram,Pundhara To Pathikashram ⇆ Pundhara To Pathika...,3,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ahmedabad,NaN,NaN,0
648,MTC-18761,T23C CT,Saidapet TO Ayanavaram,3,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,chennai,NaN,NaN,0
650,MTC-11919,55K,Tambaram West TO Guduvanchery,3,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,chennai,NaN,NaN,0
655,MTC-18763,T23C CT1,Thiruvanmiyur TO Egmore,3,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,chennai,NaN,NaN,0
658,MTC-13402,S43,Pudur TO Villivakkam,3,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,chennai,NaN,NaN,0



## 5. Investigating Routes Without Scheduled Trips

Some routes in the dataset have zero scheduled trips and zero stops. Their service hours, operating days, and route distances are also missing.

These missing values may be caused by the absence of scheduled service information rather than random data loss.

In this step, we identify routes without scheduled trips and separate them from routes with missing information that may still be usable for machine learning.


In [4]:

# Identify routes without scheduled trips
no_trip_routes = df[
    df["scheduled_trip_count"].eq(0)
]

print("Routes Without Scheduled Trips:",
      len(no_trip_routes))

print("\nRoutes Without Trips by City:")
print(no_trip_routes["city"].value_counts())

print("\nSample Routes Without Trips:")
display(
    no_trip_routes[
        [
            "route_id",
            "route_short_name",
            "city",
            "scheduled_trip_count",
            "stop_count"
        ]
    ].head(10)
)


Routes Without Scheduled Trips: 1006

Routes Without Trips by City:
city
chennai      677
mumbai       323
ahmedabad      6
Name: count, dtype: int64

Sample Routes Without Trips:


,route_id,route_short_name,city,scheduled_trip_count,stop_count
626,Kalol To Pathikashram Via Sardhav,Kalol To Pathikashram Via Sardhav,ahmedabad,0,0
627,Mansa To Pathikashram,Mansa To Pathikashram,ahmedabad,0,0
631,Pathikashram To Kalol Via Sardhav,Pathikashram To Kalol Via Sardhav,ahmedabad,0,0
632,Pathikashram To Mansa,Pathikashram To Mansa,ahmedabad,0,0
634,Pathikashram To Pundhara,Pathikashram To Pundhara,ahmedabad,0,0
637,Pundhara To Pathikashram,Pundhara To Pathikashram,ahmedabad,0,0
648,MTC-18761,T23C CT,chennai,0,0
650,MTC-11919,55K,chennai,0,0
655,MTC-18763,T23C CT1,chennai,0,0
658,MTC-13402,S43,chennai,0,0


In [5]:
no_trip_routes = df[df["scheduled_trip_count"] == 0]

print("Routes with no scheduled trips:", len(no_trip_routes))
print(no_trip_routes.groupby("city").size())

Routes with no scheduled trips: 1006
city
ahmedabad      6
chennai      677
mumbai       323
dtype: int64


In [6]:
df_model = df[df["scheduled_trip_count"] > 0].copy()

print("Original rows:", len(df))
print("Model-ready rows:", len(df_model))
print("Routes excluded:", len(df) - len(df_model))

Original rows: 8322
Model-ready rows: 7316
Routes excluded: 1006


In [7]:
missing_counts = df_model.isnull().sum()
print(missing_counts[missing_counts > 0])

route_short_name                    1031
estimated_straight_line_route_km       4
trips_with_distance                    4
dtype: int64


In [8]:
df_model[
    df_model["estimated_straight_line_route_km"].isna()
][["route_id", "city", "scheduled_trip_count", "trips_with_distance"]]

,route_id,city,scheduled_trip_count,trips_with_distance
5965,SRR-43551,chennai,1,NaN
5966,SRR-43552,chennai,1,NaN
5967,SRR-43553,chennai,1,NaN
5968,SRR-43554,chennai,1,NaN


In [9]:
df_model.loc[
    df_model["route_id"].isin(
        ["SRR-43551", "SRR-43552", "SRR-43553", "SRR-43554"]
    ),
    ["route_id", "estimated_straight_line_route_km", "trips_with_distance"]
]

,route_id,estimated_straight_line_route_km,trips_with_distance
5965,SRR-43551,NaN,NaN
5966,SRR-43552,NaN,NaN
5967,SRR-43553,NaN,NaN
5968,SRR-43554,NaN,NaN


In [10]:
df_distance = df_model.dropna(
    subset=["estimated_straight_line_route_km"]
).copy()

print("Routes with distance:", len(df_distance))
print("Routes without distance:", len(df_model) - len(df_distance))

Routes with distance: 7312
Routes without distance: 4


In [11]:
from pathlib import Path
import pandas as pd
import numpy as np

project_dir = Path(r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker")
processed_dir = project_dir / "data" / "processed"
reports_dir = project_dir / "reports"

processed_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

# Keep a copy of the distance-ready dataset
df_prepared = df_distance.copy()

# Fill missing route names with a label; do not use route names as model features
df_prepared["route_short_name"] = (
    df_prepared["route_short_name"].fillna("Unknown")
)

print("Prepared dataset shape:", df_prepared.shape)
print("Missing values remaining:", df_prepared.isnull().sum().sum())

Prepared dataset shape: (7312, 20)
Missing values remaining: 0


In [12]:
# Numerical features
numeric_features = [
    "scheduled_trip_count",
    "stop_count",
    "estimated_straight_line_route_km",
    "operating_day_count",
    "start_service_hour",
    "end_service_hour",
]

# Categorical features
categorical_features = [
    "route_type",
    "city",
]

# Day-of-week indicators
day_features = [
    "operates_monday",
    "operates_tuesday",
    "operates_wednesday",
    "operates_thursday",
    "operates_friday",
    "operates_saturday",
    "operates_sunday",
]

feature_columns = numeric_features + categorical_features + day_features

X = df_prepared[feature_columns].copy()

# Convert day indicators to numeric values
for column in day_features:
    X[column] = pd.to_numeric(X[column], errors="coerce")

print("Feature dataset shape:", X.shape)
print("Missing values in selected features:", X.isnull().sum().sum())

Feature dataset shape: (7312, 15)
Missing values in selected features: 0


In [13]:
# Fill missing numeric values with each column's median
for column in numeric_features + day_features:
    X[column] = X[column].fillna(X[column].median())

# Fill missing categories with "Unknown"
for column in categorical_features:
    X[column] = X[column].fillna("Unknown").astype(str)

print("Missing values after handling:", X.isnull().sum().sum())

Missing values after handling: 0


In [14]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", MinMaxScaler(), numeric_features + day_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ],
    remainder="drop",
)

# Fit the preprocessing steps and transform the features
X_processed = preprocessor.fit_transform(X)

# Convert the transformed data into a DataFrame
feature_names = preprocessor.get_feature_names_out()

X_processed_df = pd.DataFrame(
    X_processed.toarray() if hasattr(X_processed, "toarray") else X_processed,
    columns=feature_names,
    index=X.index,
)

print("Processed feature shape:", X_processed_df.shape)
X_processed_df.head()

Processed feature shape: (7312, 20)


,numeric__scheduled_trip_count,numeric__stop_count,numeric__estimated_straight_line_route_km,numeric__operating_day_count,numeric__start_service_hour,numeric__end_service_hour,numeric__operates_monday,numeric__operates_tuesday,numeric__operates_wednesday,numeric__operates_thursday,numeric__operates_friday,numeric__operates_saturday,numeric__operates_sunday,categorical__route_type_1,categorical__route_type_3,categorical__city_ahmedabad,categorical__city_chennai,categorical__city_hyderabad,categorical__city_mumbai,categorical__city_pune
0,0.004314,0.054156,0.062221,1.0,0.176471,0.061093,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
1,0.010067,0.099496,0.031252,1.0,0.176471,0.067524,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
2,0.006232,0.064232,0.020851,1.0,0.147059,0.073955,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
3,0.002397,0.034005,0.012503,1.0,0.352941,0.070740,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
4,0.103068,0.049118,0.089048,1.0,0.147059,0.073955,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0


In [15]:
# Save the processed feature matrix
processed_features_path = processed_dir / "module4_processed_features.csv"
X_processed_df.to_csv(processed_features_path, index=False)

# Save the route details alongside the processed features
route_details = df_prepared[
    ["route_id", "route_short_name", "city"]
].reset_index(drop=True)

processed_with_details = pd.concat(
    [route_details, X_processed_df.reset_index(drop=True)],
    axis=1,
)

processed_with_details.to_csv(
    processed_dir / "module4_processed_dataset.csv",
    index=False,
)

# Save the fitted preprocessing pipeline
pipeline_path = processed_dir / "module4_preprocessing_pipeline.pkl"
joblib.dump(preprocessor, pipeline_path)

print("Saved processed features:", processed_features_path)
print("Saved processed dataset:", processed_dir / "module4_processed_dataset.csv")
print("Saved preprocessing pipeline:", pipeline_path)

Saved processed features: C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\data\processed\module4_processed_features.csv
Saved processed dataset: C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\data\processed\module4_processed_dataset.csv
Saved preprocessing pipeline: C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\data\processed\module4_preprocessing_pipeline.pkl


In [16]:
report = f"""
# RouteRanker — Module 4: Data Preprocessing

## 1. Objective
Prepare GTFS-based route features for machine-learning analysis.

## 2. Data filtering
- Original routes: {len(df)}
- Routes with scheduled trips: {len(df_model)}
- Routes with available distance: {len(df_distance)}
- Routes excluded from the distance-ready dataset: {len(df) - len(df_distance)}

## 3. Missing-value handling
- Missing route names were labeled as "Unknown".
- Remaining missing numerical features were filled using their column median.
- Remaining missing categorical features were labeled as "Unknown".

## 4. Feature preparation
- Numerical and day-of-week features were scaled using Min-Max Scaling.
- City and route type were encoded using One-Hot Encoding.
- Route IDs and route names were kept as details, not used as model features.

## 5. Output
- Processed feature rows: {X_processed_df.shape[0]}
- Processed feature columns: {X_processed_df.shape[1]}
- Processed features: module4_processed_features.csv
- Processed dataset: module4_processed_dataset.csv
- Saved preprocessing pipeline: module4_preprocessing_pipeline.pkl

## 6. Conclusion
Module 4 prepared the route data by handling missing values, scaling numerical
features, and encoding categorical features. The processed dataset and fitted
preprocessing pipeline are saved for the next modelling module.
"""

report_path = reports_dir / "module4_data_preprocessing_report.md"
report_path.write_text(report, encoding="utf-8")

print("Module 4 report saved:", report_path)
print("Module 4 completed successfully!")

Module 4 report saved: C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\reports\module4_data_preprocessing_report.md
Module 4 completed successfully!


In [17]:
from pathlib import Path

# Project folders
project_dir = Path(r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker")
processed_dir = project_dir / "data" / "processed"
reports_dir = project_dir / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

# Check saved deliverables
required_files = [
    processed_dir / "module4_processed_features.csv",
    processed_dir / "module4_processed_dataset.csv",
    processed_dir / "module4_preprocessing_pipeline.pkl",
]

print("DELIVERABLE CHECK")
for file_path in required_files:
    print(f"{file_path.name}: {'Found' if file_path.exists() else 'MISSING'}")

# Write a report based on the processing completed
report = f"""
# RouteRanker — Module 4: Data Preprocessing

## Objective
Prepare the feature-engineered route dataset for the planned clustering stage.

## Dataset and filtering
- Original dataset: {len(df)} rows.
- Routes with scheduled trips: {len(df_model)}.
- Routes with distance available: {len(df_distance)}.
- Routes with no scheduled trips excluded from the distance-ready dataset: {len(df) - len(df_model)}.
- Routes without distance excluded from the distance-ready dataset: {len(df_model) - len(df_distance)}.

The original dataset was preserved. The distance-ready dataset was used for this preprocessing workflow.

## Missing values
- The original dataset contained missing values in route names, distance-related fields, operating-day fields, and service-hour fields.
- Routes with no scheduled trips were excluded from the distance-ready dataset because they had no scheduled service to analyze.
- Four routes had scheduled trips but no distance values; they were excluded from the distance-ready dataset used here.
- Missing route names were labeled "Unknown".
- The selected model features had {X.isnull().sum().sum()} missing values before the missing-value handling step, so no numerical imputation was required.

## Feature selection and transformation
- Selected {len(feature_columns)} features: numerical service measures, route type, city, and operating-day indicators.
- Route IDs and route names were retained as identifiers/details, not used as model features.
- Numerical and day-of-week features were scaled using Min-Max Scaling.
- City and route type were transformed using One-Hot Encoding.
- The processed feature matrix contains {X_processed_df.shape[0]} rows and {X_processed_df.shape[1]} columns.

## Training and evaluation strategy
RouteRanker is planned to use clustering. Clustering does not require target labels, so no target column or classification-style 80:20 split was created in this module. The clustering evaluation strategy will be documented in the modelling module. If a validation split is used later, preprocessing must be fitted on the training data and then applied to validation data.

## Assumptions and limitations
- Scheduled trip counts describe planned service, not actual passenger demand or overcrowding.
- Distance values are estimated straight-line route distances, not measured road distances.
- Routes excluded from this distance-ready dataset remain in the original dataset.
- No passenger counts or unsupported route-performance labels were created.

## Saved outputs
- `data/processed/module4_processed_features.csv`
- `data/processed/module4_processed_dataset.csv`
- `data/processed/module4_preprocessing_pipeline.pkl`
- `reports/module4_data_preprocessing_report.md`

## Conclusion
The distance-ready route features were checked, selected, scaled, and encoded for the next clustering stage. The original dataset was preserved, and the preprocessing outputs were saved for reproducibility.
"""

report_path = reports_dir / "module4_data_preprocessing_report.md"
report_path.write_text(report, encoding="utf-8")

print("\nUpdated report:", report_path)
print("Module 4 final report updated.")

DELIVERABLE CHECK
module4_processed_features.csv: Found
module4_processed_dataset.csv: Found
module4_preprocessing_pipeline.pkl: Found

Updated report: C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\reports\module4_data_preprocessing_report.md
Module 4 final report updated.


In [1]:
from pathlib import Path
import pandas as pd
import joblib

PROJECT_DIR = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker"
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

print("MODULE 4 FINAL VERIFICATION")
print("=" * 50)

# 1. Check files
files = [
    "module4_processed_dataset.csv",
    "module4_processed_features.csv",
    "module4_preprocessing_pipeline.pkl"
]

for name in files:
    path = PROCESSED_DIR / name
    print(name, ":", "FOUND" if path.exists() else "NOT FOUND")

# 2. Check processed features
features_path = PROCESSED_DIR / "module4_processed_features.csv"

if features_path.exists():

    df = pd.read_csv(features_path)

    print("\nDATASET CHECK")
    print("Shape:", df.shape)
    print("Missing values:", df.isnull().sum().sum())
    print("Duplicate rows:", df.duplicated().sum())

    print("\nDATA TYPES")
    print(df.dtypes)

    print("\nMINIMUM VALUES")
    print(df.min(numeric_only=True))

    print("\nMAXIMUM VALUES")
    print(df.max(numeric_only=True))

# 3. Check preprocessing pipeline
pipeline_path = PROCESSED_DIR / "module4_preprocessing_pipeline.pkl"

if pipeline_path.exists():
    pipeline = joblib.load(pipeline_path)
    print("\nPipeline loaded successfully!")
    print("Pipeline type:", type(pipeline))

MODULE 4 FINAL VERIFICATION
module4_processed_dataset.csv : FOUND
module4_processed_features.csv : FOUND
module4_preprocessing_pipeline.pkl : FOUND

DATASET CHECK
Shape: (7312, 20)
Missing values: 0
Duplicate rows: 337

DATA TYPES
numeric__scheduled_trip_count                float64
numeric__stop_count                          float64
numeric__estimated_straight_line_route_km    float64
numeric__operating_day_count                 float64
numeric__start_service_hour                  float64
numeric__end_service_hour                    float64
numeric__operates_monday                     float64
numeric__operates_tuesday                    float64
numeric__operates_wednesday                  float64
numeric__operates_thursday                   float64
numeric__operates_friday                     float64
numeric__operates_saturday                   float64
numeric__operates_sunday                     float64
categorical__route_type_1                    float64
categorical__route_type_3  

In [2]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\data\processed"
)

dataset = pd.read_csv(
    PROCESSED_DIR / "module4_processed_dataset.csv"
)

features = pd.read_csv(
    PROCESSED_DIR / "module4_processed_features.csv"
)

print("DUPLICATE ANALYSIS")
print("=" * 45)

print(
    "Processed dataset duplicates:",
    dataset.duplicated().sum()
)

print(
    "Processed feature duplicates:",
    features.duplicated().sum()
)

print("\nPROCESSED DATASET COLUMNS:")
print(dataset.columns.tolist())

print("\nFIRST 10 DUPLICATE FEATURE ROWS:")

duplicates = features[
    features.duplicated(keep=False)
]

display(duplicates.head(10))

DUPLICATE ANALYSIS
Processed dataset duplicates: 0
Processed feature duplicates: 337

PROCESSED DATASET COLUMNS:
['route_id', 'route_short_name', 'city', 'numeric__scheduled_trip_count', 'numeric__stop_count', 'numeric__estimated_straight_line_route_km', 'numeric__operating_day_count', 'numeric__start_service_hour', 'numeric__end_service_hour', 'numeric__operates_monday', 'numeric__operates_tuesday', 'numeric__operates_wednesday', 'numeric__operates_thursday', 'numeric__operates_friday', 'numeric__operates_saturday', 'numeric__operates_sunday', 'categorical__route_type_1', 'categorical__route_type_3', 'categorical__city_ahmedabad', 'categorical__city_chennai', 'categorical__city_hyderabad', 'categorical__city_mumbai', 'categorical__city_pune']

FIRST 10 DUPLICATE FEATURE ROWS:


,numeric__scheduled_trip_count,numeric__stop_count,numeric__estimated_straight_line_route_km,numeric__operating_day_count,numeric__start_service_hour,numeric__end_service_hour,numeric__operates_monday,numeric__operates_tuesday,numeric__operates_wednesday,numeric__operates_thursday,numeric__operates_friday,numeric__operates_saturday,numeric__operates_sunday,categorical__route_type_1,categorical__route_type_3,categorical__city_ahmedabad,categorical__city_chennai,categorical__city_hyderabad,categorical__city_mumbai,categorical__city_pune
514,0.049856,0.039043,0.025896,1.0,0.176471,0.073955,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
516,0.049856,0.039043,0.025896,1.0,0.176471,0.073955,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
821,0.000000,0.002519,0.003871,1.0,0.647059,0.070740,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
947,0.000000,0.002519,0.003871,1.0,0.617647,0.067524,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
967,0.000000,0.002519,0.003782,1.0,0.176471,0.019293,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
976,0.000000,0.002519,0.003871,1.0,0.647059,0.070740,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1040,0.000000,0.002519,0.003871,1.0,0.617647,0.067524,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1155,0.000000,0.002519,0.003782,1.0,0.176471,0.019293,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1162,0.000000,0.002519,0.003871,1.0,0.617647,0.067524,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1242,0.000000,0.040302,0.038813,1.0,0.176471,0.022508,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


In [3]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\data\processed"
)

df = pd.read_csv(
    PROCESSED_DIR / "module4_processed_dataset.csv"
)

# Find identical feature combinations
feature_columns = [
    col for col in df.columns
    if col.startswith(("numeric__", "categorical__"))
]

duplicate_mask = df.duplicated(
    subset=feature_columns,
    keep=False
)

duplicate_routes = df.loc[
    duplicate_mask,
    ["route_id", "route_short_name", "city"] + feature_columns
]

print("ROUTES WITH IDENTICAL FEATURES")
print("=" * 45)

print("Number of affected routes:", len(duplicate_routes))
print("Number of unique route IDs:",
      duplicate_routes["route_id"].nunique())

print("\nCITY-WISE DUPLICATE FEATURE RECORDS:")
print(duplicate_routes.groupby("city").size())

print("\nSAMPLE:")
display(duplicate_routes.head(10))

ROUTES WITH IDENTICAL FEATURES
Number of affected routes: 502
Number of unique route IDs: 502

CITY-WISE DUPLICATE FEATURE RECORDS:
city
ahmedabad      2
chennai      500
dtype: int64

SAMPLE:


,route_id,route_short_name,city,numeric__scheduled_trip_count,numeric__stop_count,numeric__estimated_straight_line_route_km,numeric__operating_day_count,numeric__start_service_hour,numeric__end_service_hour,numeric__operates_monday,...,numeric__operates_friday,numeric__operates_saturday,numeric__operates_sunday,categorical__route_type_1,categorical__route_type_3,categorical__city_ahmedabad,categorical__city_chennai,categorical__city_hyderabad,categorical__city_mumbai,categorical__city_pune
514,2D-OI,2D-OI,ahmedabad,0.049856,0.039043,0.025896,1.0,0.176471,0.073955,1.0,...,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
516,2U-IO,2U-IO,ahmedabad,0.049856,0.039043,0.025896,1.0,0.176471,0.073955,1.0,...,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
821,MTC-11953,557M,chennai,0.000000,0.002519,0.003871,1.0,0.647059,0.070740,1.0,...,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
947,MTC-11989,558C,chennai,0.000000,0.002519,0.003871,1.0,0.617647,0.067524,1.0,...,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
967,MTC-12005,558P,chennai,0.000000,0.002519,0.003782,1.0,0.176471,0.019293,1.0,...,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
976,MTC-12013,558P,chennai,0.000000,0.002519,0.003871,1.0,0.647059,0.070740,1.0,...,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1040,MTC-11939,557C,chennai,0.000000,0.002519,0.003871,1.0,0.617647,0.067524,1.0,...,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1155,MTC-18093,592V,chennai,0.000000,0.002519,0.003782,1.0,0.176471,0.019293,1.0,...,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1162,MTC-18092,592V,chennai,0.000000,0.002519,0.003871,1.0,0.617647,0.067524,1.0,...,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
1242,MTC-17105,78,chennai,0.000000,0.040302,0.038813,1.0,0.176471,0.022508,1.0,...,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


In [4]:
import joblib
from pathlib import Path

pipeline_path = Path(
    r"C:\Users\LENOVO\OneDrive\Desktop\RouteRanker\data\processed\module4_preprocessing_pipeline.pkl"
)

pipeline = joblib.load(pipeline_path)

print("Pipeline loaded successfully!")
print("Pipeline type:", type(pipeline))

Pipeline loaded successfully!
Pipeline type: <class 'sklearn.compose._column_transformer.ColumnTransformer'>
